# These are the examples and exercises for Lesson 2: "Topic Modelling".

In [2]:
import string
from collections import Counter
from pprint import pprint
import gzip
import matplotlib.pyplot as plt 
import numpy as np
from numpy.linalg import svd

%matplotlib inline

Fetch stop words from a standard set; save them in a set object to remove duplicates from the source and to make lookups quick.

In [3]:
# The mode flags for the `open` function are "r": read, "t": text mode
stopwords = set([word.lower().strip() for word in open("data/nltk_stopwords.txt", "rt").readlines()])

For extracting words this time, we need to keep hashtags and mentions; and remove stop words.

In [4]:
def extract_words(text, stopwords):
    temp = text.split() # Split the text on whitespace
    text_words = []

    punctuation = set(string.punctuation)
    
    #Keep #tags and @mentions
    punctuation.remove("#")
    punctuation.remove("@")
    
    for word in temp:
        # Remove any punctuation characters present in the beginning of the word
        while len(word) > 0 and word[0] in punctuation:
            word = word[1:]

        # Remove any punctuation characters present in the end of the word
        while len(word) > 0 and word[-1] in punctuation:
            word = word[:-1]

        # Simple rule to eliminate (most) URLs
        if len(word) > 0 and "/" not in word:
            # If it's not a stopword
            if word.lower() not in stopwords:
                # Append this word into our list of words.
                text_words.append(word.lower())

    return text_words


Process tweet data from the CSV file that contains references to Apple. This will be the corpus we will perform analysis on.

In [5]:
tweets = []
line_count = 0

for line in open("data/Apple-Twitter-Sentiment-DFE.csv", "rt"):
    fields = line.strip().split(',')
    
    line_count += 1
    
    # Skip the first line of the file which contains the header
    if line_count == 1:
        continue
    
    text = ",".join(fields[11:])
    
    if len(text) == 0:
        continue
    
    words = extract_words(text, stopwords)
    
    if len(words) > 0:
        tweets.append(words)

    # Limit the corpus
    if len(tweets) == 200:
        break

Define the function to calculate the Inverse Document Frequency for each word and the TFIDF matrix.

In [6]:
def inv_doc_freq(corpus_words):
    number_docs = len(corpus_words)
    
    document_count = {}

    for document in corpus_words:
        word_set = set(document)

        for word in word_set:
            document_count[word] = document_count.get(word, 0) + 1
    
    IDF = {}
    
    for word in document_count:
        IDF[word] = np.log(number_docs/document_count[word])
        
    
    return IDF

def tf_idf(corpus_words):
    IDF = inv_doc_freq(corpus_words)
    
    TFIDF = []

    # Counter is a subclass of `dict` for counting hashable objects
    for document in corpus_words:
        TFIDF.append(Counter(document))
    
    for document in TFIDF:
        for word in document:
            document[word] = document[word]*IDF[word]
            
    return TFIDF

Note that while we call it a matrix, this is effectively a list of dictionaries, which we can consider to be a sparse representation of a matrix.

In [7]:
TFIDF = tf_idf(tweets)

In [8]:
def build_vocabulary(TFIDF):
    words = set()
    
    for document in TFIDF:
        # this operation is union of words and document.keys() and self-assign
        # document.keys() is a set (set operators can only work with set types)
        words |= document.keys()
    
    word_list = list(words)

    # The zip function will generate a tuple for each word that is matched with the index generated by the range function
    word_dict = dict(zip(word_list, range(len(word_list))))
    
    return word_dict, word_list

In [9]:
word_dict, word_list = build_vocabulary(TFIDF)

In [10]:
vocabulary_size = len(word_dict)
print(f"We have {vocabulary_size} words in our vocabulary")

We have 927 words in our vocabulary


Now use the TFIDF matrix and our vocabulary to generate the Term Document matrix. This is just a matter of rearranging the values in our (sparse) TFIDF matrix into the full term document matrix.

In [11]:
def term_document_matrix(TFIDF, word_list, word_dict):
    vocabulary_size = len(word_dict)
    number_documents = len(TFIDF)

    # Generate a zero-filled matrix on the specified dimensions
    matrix = np.zeros((vocabulary_size, number_documents))
    
    for doc_index in range(number_documents):
        document = TFIDF[doc_index]
        
        for word in document.keys():
            position = word_dict[word]
            
            matrix[position, doc_index] = document[word]
            
    return matrix

In [12]:
TDM = term_document_matrix(TFIDF, word_list, word_dict)
print("Our dataset has:\n%u unique words\n%u documents"%(TDM.shape))

Our dataset has:
927 unique words
200 documents


## Lesson 2.2: Explicit semantic analysis

The similarity of words or new documents can be measured using the [cosine similarity](https://en.wikipedia.org/wiki/Cosine_similarity).

[Explict semantic analysis](https://en.wikipedia.org/wiki/Explicit_semantic_analysis), despite its simplicity based on this mathematical relationship, has been shown to improve the performance of search systems. However, ESA requires the use of a large corpus as a knowledge base (such as the entire English Wikipedia) resulting in high dimensional representations of words and documents, thus requiring vast amounts of computing resources.

In ESA we use the TD matrix of our corpus as a knowledge base that we can use to look up related documents.

In [13]:
# a document that already has stop words removed
new_tweet = ['macbook', 'mini', 'rocket']

The `find_related_docs` simply calcuates the vector corresponding to the new "document" and returns a list of the corresponding weights sorted in decreasing order.

In [14]:
def find_related_docs(tweet, TDM):
    new_vector = np.zeros(TDM.shape[1])
    
    for word in tweet:
        pos = word_dict[word]
        new_vector += TDM[pos, :]
        
    # Now the entries of new_vector tell us which documents are activated by this one.
    # Let's extract the list of documents sorted by activation
    doc_list = sorted(zip(range(TDM.shape[1]), new_vector), key=lambda x:x[1], reverse=True)
    
    return doc_list

In [20]:
related = find_related_docs(new_tweet, TDM)

# Show top 5 results
for tweet_index, score in related[:5]:
    print(f"{tweet_index}\t{score:4.2f}\t{" ".join(tweets[tweet_index])}")

17	6.73	jh hines staff newly issued @apple #connected macbook ipad mini #txed
34	6.73	rt @tra_hall jh hines staff newly issued @apple #connected macbook ipad mini #txed #txed
2	5.30	#aapl:5 rocket stocks buy december gains apple
166	3.51	ipad mini unboxing via @youtube @apple #ipadmini #ipad #macbook #macbookpro #startup #hipster #unboxing
167	3.51	ipad mini first time startup via @youtube @apple #ipadmini #ipad #macbook #macbookpro #startup #hipster #unbox


Let's measure the similarity.

In [21]:
def similarity(vec1, vec2):
    # Generate the dot product of the two vectors
    sim = np.dot(vec1, vec2)
    norm1 = np.sqrt(np.dot(vec1, vec1))
    norm2 = np.sqrt(np.dot(vec2, vec2))

    # Fancy vector math
    return sim/(norm1*norm2)

In [22]:
def find_similar_words(tweet, TDM):
    new_vector = np.zeros(TDM.shape[1])
    
    for word in tweet:
        pos = word_dict[word]
        new_vector += TDM[pos, :]
    
    sim = [similarity(new_vector, TDM[i, :]) for i in range(TDM.shape[0])]
    
    sim_words = sorted(zip(range(TDM.shape[0]), sim), key=lambda x:x[1], reverse=True)
    
    return sim_words

In [24]:
similar = find_similar_words(new_tweet, TDM)

for word_index, score in similar[:5]:
    print(f"{word_list[word_index]}\t{score:5.2f}")

macbook	 0.76
mini	 0.74
ipad	 0.68
#macbook	 0.66
@youtube	 0.63


## Lesson 2.4: Implement latent semantic analysis

[Singular Value Decomposition](https://en.wikipedia.org/wiki/Singular_value_decomposition) (SVD): Decompose a matrix (M), such as the term document matrix (TDM) above, into three matrices. If M has the dimensions `m` x `n`, then the resulting matrices has these shapes: U (`m` x `m`), sigma (`m` x `n`), Vt (`n` x `n`).

Latent Semantic Analysis (LSA): LSA attempts to implicitly determine a "latent" reperesentation for each word and document. 

In [25]:
# The SVD function is found in NumPy's Linear Algebra module
u, sigma, vt = svd(TDM)

# And the resultant shapes:
print(TDM.shape)
print(u.shape, sigma.shape, vt.shape)

(927, 200)
(927, 927) (200,) (200, 200)


Reduce the internal dimensions of the matrices, which will preserve the most significant latent dimensions of the dataset (higher sigma value). This will reveal the dominant topics in the data.

In [27]:
k = 10

# Convert the vector of singular values into a diagonal matrix
sigma_k = sigma[:k]
Sk = np.diag(sigma_k)

# Drop the extraneous dimensions in the other two matrices.
u_k = u[:,:k]
vt_k = vt[:k, :]

print(u_k.shape, Sk.shape, vt_k.shape)

(927, 10) (10, 10) (10, 200)


In [28]:
# Sort the list of words by the weight they have in a specific topic
def top_words(vector, word_list):
    doc_list = sorted(zip(word_list, vector), key=lambda x:x[1], reverse=True)
    
    return doc_list

In [29]:
topic_words = top_words(u_k[:,3], word_list)
for word, weight in topic_words[:10]:
    print(word, weight)

day 0.2874648801433035
aids 0.28262164097539044
world 0.2692253682039429
red 0.2647556149261591
logo 0.229034060704606
#apple 0.18504347302677993
apple 0.13868223906604218
dec 0.12638896410764353
launches 0.12638896410764353
abl 0.12638896410764353


In [30]:
document = ['ipad', 'mini', 'price']

doc_vector = np.zeros(vocabulary_size)
for word in document:
    doc_vector[word_dict[word]] += 1

doc_singular = 1 / sigma_k * np.dot(u_k.T, doc_vector)
pprint(doc_singular)

array([ 1.62243400e-04,  1.44900491e-02, -1.09973213e-03,  4.30257980e-05,
       -9.69106965e-04,  4.54646796e-03, -7.58613138e-04,  2.40772326e-03,
       -3.82599561e-03,  1.95924079e-03])


In [31]:
# `argmax` returns the indices of the maximum values along an axis
topic = np.argmax(doc_singular)

topic_words = top_words(u_k[:,topic], word_list)
for word, weight in topic_words[:10]:
    print(word, weight)

#macbook 0.3072355938213899
@youtube 0.30540145699726656
#hipster 0.30540145699726656
#startup 0.30540145699726656
#macbookpro 0.2980960628908024
via 0.29187957210987736
mini 0.2094966623380365
ipad 0.20381766001212515
#ipadmini 0.19891025664321707
startup 0.1932748800761493


## Lesson 2.5: Non-negative matrix factorization

In [NMF](https://en.wikipedia.org/wiki/Non-negative_matrix_factorization), a matrix V is factorized into two matrices W and H, with the property that all three matrices have no negative elements. This non-negativity makes the resulting data easier to determine latent dimensions.

In [36]:
def NMF(V, k):
    n, m = V.shape

    W = np.random.rand(n, k)
    H = np.random.rand(k, m)

    error = 1
    err_rate = 1

    while err_rate > 1e-4:
        prev_error = error

        hn = np.dot(W.T, V)
        hd = np.dot(W.T, np.dot(W, H))

        H = H*hn/hd

        wn = np.dot(V, H.T)
        wd = np.dot(W, np.dot(H, H.T))

        W = W*wn/wd

        error = np.sum(np.power(V - np.dot(W, H), 2.0))
        err_rate = np.abs(prev_error - error)

    return W, H, error

When applying NMF to the term document matrix (TDM), the results prove useful for topic detection. The W matrix provides us with the definition of each topic as a weighted distribution over all the words in the corpus.

In [37]:
W, H, err = NMF(TDM, 30)

In [38]:
topic_words = top_words(W[:,0], word_list)
for word, weight in topic_words[:10]:
    print(word, weight)

@twtr 24.90358724232145
follow 24.90358724232145
@linkd 24.90358724232145
see 24.90358724232145
3-6 24.90358724232145
whether 24.90358724232145
#nasdaq 24.90358724232145
fell 24.90358724232145
@baba 24.90358724232145
@fb 24.90358724232145


In [39]:
topic_words = top_words(W[:,10], word_list)
for word, weight in topic_words[:10]:
    print(word, weight)

thanks 38.98173175097114
dumb 20.977234509334327
non 20.977234509334327
drops 20.977234509334327
points 20.977234509334327
@applenws 20.977234509334327
followers 20.977234509334327
dummies 20.977234509334327
minute 20.977234509334327
#rumors 20.977234509334327


The code above provide a "good enough" presentation of the algorithms used for factorization. It is a preferable practice to utilise open-source libraries to ensure the functionality works with guards and edge-cases.